# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a structured walk-through for loading and exploring the FAIR^2 colorectal cancer dataset using the [`mlcroissant`](https://mlcroissant.readthedocs.io/) library. 

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the FAIR^2 dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available RecordSets, their Fields, and the corresponding `@id`s. Only the `@id` fields are used for referencing.

Below, we examine what record sets are available in the dataset, as well as the field structure in each.

In [ ]:
# List available record sets with their @id and field ids

record_sets = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    # If recordSet is a single object, make it a list
    if isinstance(metadata.recordSet, dict):
        record_set_objs = [metadata.recordSet]
    else:
        record_set_objs = metadata.recordSet

    for record_set in record_set_objs:
        record_set_id = record_set['@id'] if isinstance(record_set, dict) and '@id' in record_set else record_set
        print(f"Found RecordSet: {record_set_id}")
        record_sets.append(record_set_id)
        # Try to print fields of this record set, if available
        fields = None
        if isinstance(record_set, dict) and 'field' in record_set:
            fields = record_set['field']
        if fields:
            if isinstance(fields, dict):
                fields = [fields]
            print("  Fields:")
            for field in fields:
                field_id = field['@id'] if isinstance(field, dict) and '@id' in field else field
                print(f"    - {field_id}")
else:
    # fallback: try discovering record sets by inspecting dataset.records()
    from mlcroissant.dataset.dataset import _get_record_set_ids
    record_sets = _get_record_set_ids(dataset._graph)
    print("Record Sets discovered:")
    for rsid in record_sets:
        print(f" - {rsid}")
        # Show their fields
        recset = dataset._graph.get(rsid, {})
        if 'cr:field' in recset:
            fields = recset['cr:field']
            if not isinstance(fields, list):
                fields = [fields]
            print("  Fields:")
            for fid in fields:
                print(f"    - {fid}")

## 3. Data Extraction
Load tabular data from the available record set(s) into Pandas DataFrames for exploration. Use their `@id` discovered above.

_**Note:** Since the schema exposes a single main clinical table, we use that RecordSet's `@id`._

In [ ]:
# Usually there will be one main clinical table for this dataset. Find its @id.

# If not found automatically above, set manually:
clinical_record_set_id = None
if record_sets:
    clinical_record_set_id = record_sets[0]  # typically only one
# In case record_sets is empty, set to known id (from schema):
if not clinical_record_set_id:
    clinical_record_set_id = 'https://api.app.sen.science/frontiers/7862866/clinical-records'  # example, update if needed

dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

print(f"Fields in main RecordSet ({clinical_record_set_id}):")
print(dataframes[clinical_record_set_id].columns.tolist())
dataframes[clinical_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
We apply some standard data processing steps: filtering a numeric field, normalizing it, and aggregating data by a key clinical variable (e.g., sex or anatomical site).

You can choose the `@id` of a numeric field (e.g., 'age_at_second_crc_diagnosis') and a grouping field (e.g., 'sex'). The IDs must correspond to the fields you printed above.

In [ ]:
# Choose the correct field @id based on the DataFrame columns available above.
# Example field IDs (replace with those found in your dataset):
numeric_field_id = None
group_field_id = None

# Try to find likely numeric field candidates:
likely_numeric = [col for col in dataframes[clinical_record_set_id].columns if 'age' in col.lower() or 'interval' in col.lower()]
print("Candidate numeric fields:", likely_numeric)
if likely_numeric:
    numeric_field_id = likely_numeric[0]  # e.g. 'age_at_second_crc_diagnosis'

# For grouping, look for 'sex', 'anatomical_site', or similar
candidate_group = [col for col in dataframes[clinical_record_set_id].columns if 'sex' in col.lower() or 'site' in col.lower() or 'location' in col.lower()]
print("Candidate grouping fields:", candidate_group)
if candidate_group:
    group_field_id = candidate_group[0]

# Now, perform EDA only if we have the numeric field
if numeric_field_id:
    # Remove missing values just in case
    df = dataframes[clinical_record_set_id].copy()
    df = df[pd.to_numeric(df[numeric_field_id], errors='coerce').notnull()]
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean()

    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.1f}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id, as_index=False)[numeric_field_id].mean()
        print(f"\nGrouped data by {group_field_id}:")
        print(grouped_df)
else:
    print("No numeric field found for demonstration.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field and its normalized values, both for the full table and for filtered records. We'll also show the group-wise mean if grouping is possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(10, 4))
    sns.histplot(df[numeric_field_id], kde=True, bins=10, color='skyblue')
    plt.axvline(df[numeric_field_id].mean(), color='red', linestyle='--', label='Mean')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.legend()
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print('Cannot plot as numeric field was not identified.')

## 6. Conclusion

We have:
- Loaded and explored the FAIR^2 colorectal cancer dataset using the Croissant schema and the `mlcroissant` library.
- Identified the available record sets and fields via their `@id`s.
- Loaded record data into DataFrames and performed simple exploratory analysis and visualizations.

**Next steps**: Further domain exploration could include more advanced filtering, statistical tests, or predictive modeling with the available fields. Always remember to refer to field and record set `@id`s for robust schema-driven data access.